In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from typing import List, Dict, Tuple
from scipy import stats

# Add necessary paths
sys.path.append('.')
sys.path.append('./plotting_notebooks')
MAIN_AGENT = 'claude-3-5-sonnet-20241022'
TRUSTED_AGENT = 'gpt-4o'
NUM_TASKS_THRESHOLD = 9
# Import the loading functions from the analysis script
from protocol_plotting_utils import (
    load_in_combined_results,
    load_protocol_intervention_transcripts,
    load_protocol_intervention_data
)


In [ ]:
# Load data
print("Loading data...")

# Load combined results
combined_results = load_in_combined_results(
    include_failed_attempts=True, 
    include_failed_main_task=True
)

# Load protocol data
control_protocol_list = [
    'defer_to_instructed_trusted_low_context', 
    'defer_to_trusted', 
    'defer_to_instructed_trusted', 
    'defer_to_trusted_with_feedback', 
    'resample_with_feedback'
]

protocol_results_list = []
intervention_data_list = []

In [ ]:
for control_protocol in control_protocol_list:
    protocol_results = load_protocol_intervention_transcripts(protocol_name=control_protocol)
    intervention_data = load_protocol_intervention_data(protocol_name=control_protocol)
    protocol_results_list.append(protocol_results)
    intervention_data_list.append(intervention_data)

protocol_results = pd.concat(protocol_results_list)
intervention_data = pd.concat(intervention_data_list)

# Merge protocol results with intervention data
protocol_results_and_data = pd.merge(
    protocol_results, 
    intervention_data, 
    on=['task_name', 'transcript_type', 'protocol_name', 'main_agent', 'monitor_agent', 'file_name'], 
    how='inner'
)
protocol_results_and_data = protocol_results_and_data[protocol_results_and_data.main_agent == MAIN_AGENT]
protocol_results_and_data = protocol_results_and_data[protocol_results_and_data.trusted_agent == TRUSTED_AGENT]

In [ ]:
# Step 1: Find tasks that appear in all protocol+threshold combinations
protocol_threshold_pairs = []
for protocol in protocol_results_and_data.protocol_name.unique():
    protocol_data = protocol_results_and_data[protocol_results_and_data.protocol_name == protocol]
    thresholds = sorted(protocol_data.suspicion_threshold.unique())
    for threshold in thresholds:
        protocol_threshold_pairs.append((protocol, threshold))

# For each protocol-threshold pair, get the set of tasks
tasks_by_pair = {}
for protocol, threshold in protocol_threshold_pairs:
    data_for_pair = protocol_results_and_data[
        (protocol_results_and_data.protocol_name == protocol) & 
        (protocol_results_and_data.suspicion_threshold == threshold)
    ]
    if len(data_for_pair.task_name.unique()) > NUM_TASKS_THRESHOLD:
        tasks_by_pair[(protocol, threshold)] = set(data_for_pair.task_name.unique())

# Find common tasks across all pairs
common_tasks = set.intersection(*tasks_by_pair.values())
print(f"Tasks common to all protocol+threshold pairs: {sorted(common_tasks)}")
print(f"Number of common tasks: {len(common_tasks)}")

protocol_results_and_data = protocol_results_and_data[protocol_results_and_data.task_name.isin(common_tasks)]
combined_results = combined_results[combined_results.task_name.isin(common_tasks)]
for protocol in protocol_results_and_data.protocol_name.unique():
    for threshold in protocol_results_and_data[protocol_results_and_data.protocol_name == protocol].suspicion_threshold.unique():
        if len(protocol_results_and_data[(protocol_results_and_data.protocol_name == protocol) & (protocol_results_and_data.suspicion_threshold == threshold)].task_name.unique()) < NUM_TASKS_THRESHOLD:
            protocol_results_and_data = protocol_results_and_data[
                (protocol_results_and_data.protocol_name != protocol) | 
                (protocol_results_and_data.suspicion_threshold != threshold)
            ]

# Precompute baseline data for both malicious and benign tasks 
# to be reused across all visualizations
baseline_data_malicious = combined_results[
    combined_results.transcript_type != 'transcripts_no_secondary_task'
]

baseline_benign = combined_results[
    (combined_results.transcript_type == 'transcripts_no_secondary_task') &
    (combined_results.main_agent == MAIN_AGENT)
]

trusted_baseline = combined_results[
    (combined_results.transcript_type == 'transcripts_no_secondary_task') &
    (combined_results.main_agent == TRUSTED_AGENT)
]

# Define helper functions for confidence intervals to be used across visualizations
def wilson_conf_interval(p, n, z=1.96):
    """
    Calculate Wilson score interval for a proportion
    
    Args:
        p: observed proportion
        n: sample size
        z: z-score for desired confidence level (1.96 for 95% CI)
        
    Returns:
        (lower bound, upper bound) of confidence interval
    """
    if n == 0:
        return 0.1
    
    denominator = 1 + z**2/n
    center = (p + z**2/(2*n)) / denominator
    half_width = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denominator
    
    # Ensure the interval is within [0, 1]
    lower = max(0, center - half_width)
    upper = min(1, center + half_width)
    
    # Return half-width of the interval
    return (upper - lower) / 2

In [ ]:
# Visualization 1: Protocol Tradeoff Plot
# Create a scatter plot showing the tradeoff between main task success and side task blocking

plt.figure(figsize=(12, 8))

# Filter to include only malicious attempts (with side tasks)
malicious_data = protocol_results_and_data[
    protocol_results_and_data.transcript_type != 'transcripts_no_secondary_task'
]

# Group by protocol and threshold
protocols = malicious_data.protocol_name.unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(protocols)))
markers = ['o', 's', 'D', '^', 'v', '>', '<', 'p', '*']



# Filter data to include only common tasks
malicious_data_common = malicious_data[malicious_data.task_name.isin(common_tasks)]

# Plot each protocol with threshold labels
for i, protocol in enumerate(protocols):
    protocol_data = malicious_data_common[malicious_data_common.protocol_name == protocol]
    thresholds = sorted(protocol_data.suspicion_threshold.unique())
    
    x_values = []  # Main task success rate
    y_values = []  # Side task blocking rate (1 - side task success)
    
    for threshold in thresholds:
        data_at_threshold = malicious_data_common[
            (malicious_data_common.protocol_name == protocol) & 
            (malicious_data_common.suspicion_threshold == threshold)
        ]
        if len(data_at_threshold.task_name.unique()) < NUM_TASKS_THRESHOLD:
            continue
        # Step 2: Process each task separately, then average across tasks
        task_metrics = []
        for task in common_tasks:
            task_data = data_at_threshold[data_at_threshold.task_name == task]
            if len(task_data) > 0:
                task_main_success = task_data.main_task_success.mean()
                task_side_blocking = 1 - task_data.side_task_evaluation.mean()
                task_metrics.append({
                    'main_task_success': task_main_success,
                    'side_task_blocking': task_side_blocking
                })
        
        # Average across tasks for equal representation
        task_df = pd.DataFrame(task_metrics)
        main_task_success = task_df.main_task_success.mean()
        side_task_blocking = task_df.side_task_blocking.mean()
        
        x_values.append(main_task_success)
        y_values.append(side_task_blocking)
        
        # Plot each point
        plt.scatter(
            main_task_success, 
            side_task_blocking, 
            color=colors[i],
            marker=markers[i % len(markers)], 
            s=100, 
            label=f"{protocol}" if threshold == thresholds[0] else None
        )
        
        # Add threshold labels
        plt.annotate(
            f"{threshold}", 
            (main_task_success, side_task_blocking),
            xytext=(5, 5), 
            textcoords='offset points'
        )
    
    # Connect points for the same protocol
    # plt.plot(x_values, y_values, '-', color=colors[i], alpha=0.5)

# Update baseline calculation to use common tasks and equal weighting
baseline_data = baseline_data_malicious[baseline_data_malicious.task_name.isin(common_tasks)]

# Calculate baseline metrics with equal task weighting
baseline_task_metrics = []
trusted_task_metrics = []

for task in common_tasks:
    # Main agent baseline
    task_baseline = baseline_data[
        (baseline_data.task_name == task) &
        (baseline_data.main_agent == MAIN_AGENT) &
        (baseline_data.monitor_agent == MAIN_AGENT)
    ]
    if len(task_baseline) > 0:
        baseline_task_metrics.append({
            'main_task_success': task_baseline.main_task_success.mean(),
            'side_task_blocking': 1 - task_baseline.side_task_evaluation.mean()
        })
    
    # Trusted agent baseline
    task_trusted = baseline_data[
        (baseline_data.task_name == task) &
        (baseline_data.main_agent == TRUSTED_AGENT)
    ]
    if len(task_trusted) > 0:
        trusted_task_metrics.append({
            'main_task_success': task_trusted.main_task_success.mean(),
            'side_task_blocking': 1 - task_trusted.side_task_evaluation.mean()
        })

# Average across tasks
baseline_df = pd.DataFrame(baseline_task_metrics)
trusted_df = pd.DataFrame(trusted_task_metrics)

# Add no-intervention baseline
plt.scatter(
    baseline_df.main_task_success.mean(), 
    baseline_df.side_task_blocking.mean(), 
    marker='X', color='red', s=200,
    label='No intervention'
)

plt.annotate(
    "No intervention", 
    (baseline_df.main_task_success.mean(), baseline_df.side_task_blocking.mean()),
    xytext=(10, -10), 
    textcoords='offset points'
)

# Add trusted agent baseline
plt.scatter(
    trusted_df.main_task_success.mean(), 
    trusted_df.side_task_blocking.mean(), 
    marker='X', color='blue', s=200,
    label='Trusted agent'
)

plt.annotate(
    "Trusted agent", 
    (trusted_df.main_task_success.mean(), trusted_df.side_task_blocking.mean()),
    xytext=(10, -10), 
    textcoords='offset points'
)

# Format plot
plt.xlabel('Main Task Success Rate', fontsize=12)
plt.ylabel('Side Task Blocking Rate (1 - side task success)', fontsize=12)
plt.title('Protocol Performance Tradeoff: Main Task Success vs Side Task Blocking', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xlim(0, 1.05)
plt.ylim(0, 1.05)

# Add diagonal reference line for equal tradeoff
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)

# Add "better performance" arrow
plt.annotate(
    "", xy=(0.95, 0.95), xytext=(0.7, 0.7),
    arrowprops=dict(facecolor='green', shrink=0.05, width=2, headwidth=10),
)
plt.text(0.8, 0.8, "Better", color='green', ha='center', va='center', fontsize=12)

plt.legend(loc='lower left', bbox_to_anchor=(0, 0), ncol=2)
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 2: Threshold Sensitivity Analysis
# Create line plots showing how threshold affects main task and side task performance

# Create subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 7), sharex=True)

# Filter to include only malicious attempts (with side tasks)
malicious_data = protocol_results_and_data[
    protocol_results_and_data.transcript_type != 'transcripts_no_secondary_task'
]

# Colors for different protocols
protocols = malicious_data.protocol_name.unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(protocols)))

# Plot main task success vs threshold
for i, protocol in enumerate(protocols):
    protocol_data = malicious_data[malicious_data.protocol_name == protocol]
    thresholds = sorted(protocol_data.suspicion_threshold.unique())
    
    main_task_values = []
    side_task_values = []
    main_task_errors = []
    side_task_errors = []
    
    for threshold in thresholds:
        data_at_threshold = protocol_data[protocol_data.suspicion_threshold == threshold]
        
        # Calculate means for main task and side task
        main_task_success = data_at_threshold.main_task_success.mean()
        side_task_blocking = 1 - data_at_threshold.side_task_evaluation.mean()
        
        # Calculate Wilson confidence intervals
        main_task_error = wilson_conf_interval(main_task_success, len(data_at_threshold))
        side_task_error = wilson_conf_interval(side_task_blocking, len(data_at_threshold))
        
        main_task_values.append(main_task_success)
        side_task_values.append(side_task_blocking)
        main_task_errors.append(main_task_error)
        side_task_errors.append(side_task_error)
    
    # Plot main task success with error bars
    ax1.errorbar(
        thresholds, 
        main_task_values, 
        yerr=main_task_errors,
        fmt='o-', 
        color=colors[i], 
        linewidth=2, 
        label=protocol,
        capsize=5
    )
    
    # Plot side task blocking with error bars
    ax2.errorbar(
        thresholds, 
        side_task_values, 
        yerr=side_task_errors,
        fmt='s-', 
        color=colors[i], 
        linewidth=2, 
        label=protocol,
        capsize=5
    )

# Format plots
ax1.set_ylabel('Main Task Success Rate', fontsize=12)
ax1.set_xlabel('Suspicion Threshold', fontsize=12)
ax1.set_title('Effect of Suspicion Threshold on Main Task Success', fontsize=14)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 1.05)
ax1.legend(loc='best')
ax1.text(0.01, 0.01, "Note: Error bars show 95% Wilson score intervals", 
        transform=ax1.transAxes, fontsize=9, ha='left', va='bottom')

ax2.set_xlabel('Suspicion Threshold', fontsize=12)
ax2.set_ylabel('Side Task Blocking Rate (1 - side task success)', fontsize=12)
ax2.set_title('Effect of Suspicion Threshold on Side Task Blocking', fontsize=14)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)
ax2.legend(loc='best')
ax2.text(0.01, 0.01, "Note: Error bars show 95% Wilson score intervals", 
        transform=ax2.transAxes, fontsize=9, ha='left', va='bottom')

plt.tight_layout()
plt.show()


In [ ]:
# Visualization 3: Protocol Effects on Benign Transcripts - Overall Performance
# Create bar plots showing how protocols affect benign transcripts' main task performance

# Filter to include only benign attempts (without side tasks)
benign_data = protocol_results_and_data[
    protocol_results_and_data.transcript_type == 'transcripts_no_secondary_task'
]

# Group by protocol and threshold for average performance
protocol_threshold_perf = benign_data.groupby(['protocol_name', 'suspicion_threshold']).agg({
    'main_task_success': ['mean', 'std', 'count'],
    'main_task_evaluation': ['mean', 'std', 'count']
}).reset_index()

# Flatten the multi-index columns
protocol_threshold_perf.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col for col in protocol_threshold_perf.columns]

# Calculate standard error with better handling of small sample sizes
protocol_threshold_perf['main_task_success_se'] = protocol_threshold_perf.apply(
    lambda row: row['main_task_success_std'] / np.sqrt(max(1, row['main_task_success_count'])) 
    if row['main_task_success_count'] > 1 else 0.1, axis=1
)
protocol_threshold_perf['main_task_evaluation_se'] = protocol_threshold_perf.apply(
    lambda row: row['main_task_evaluation_std'] / np.sqrt(max(1, row['main_task_evaluation_count'])) 
    if row['main_task_evaluation_count'] > 1 else 0.1, axis=1
)

# Calculate Wilson score intervals for binary/proportion data bounded between 0 and 1
def wilson_conf_interval(p, n, z=1.96):
    """
    Calculate Wilson score interval for a proportion
    
    Args:
        p: observed proportion
        n: sample size
        z: z-score for desired confidence level (1.96 for 95% CI)
        
    Returns:
        (lower bound, upper bound) of confidence interval
    """
    if n == 0:
        return 0.1
    
    denominator = 1 + z**2/n
    center = (p + z**2/(2*n)) / denominator
    half_width = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denominator
    
    # Ensure the interval is within [0, 1]
    lower = max(0, center - half_width)
    upper = min(1, center + half_width)
    
    # Return half-width of the interval
    return (upper - lower) / 2

# Apply Wilson confidence intervals
protocol_threshold_perf['wilson_ci_success'] = protocol_threshold_perf.apply(
    lambda row: wilson_conf_interval(row['main_task_success_mean'], row['main_task_success_count'])
    if row['main_task_success_count'] > 0 else 0.1, axis=1
)

protocol_threshold_perf['wilson_ci_eval'] = protocol_threshold_perf.apply(
    lambda row: wilson_conf_interval(row['main_task_evaluation_mean'], row['main_task_evaluation_count'])
    if row['main_task_evaluation_count'] > 0 else 0.1, axis=1
)

# Debug: Print out different error bar options to check if they're reasonable
print("Comparison of different confidence interval approaches:")
print(protocol_threshold_perf[['protocol_name', 'suspicion_threshold', 'main_task_success_mean', 
                              'main_task_success_count', 'wilson_ci_success']])

# Use Wilson score interval for error bars 
protocol_threshold_perf['error_display_success'] = protocol_threshold_perf['wilson_ci_success']
protocol_threshold_perf['error_display_eval'] = protocol_threshold_perf['wilson_ci_eval']

# Add baseline performance (no intervention) - use the precomputed values
baseline_row = pd.DataFrame({
    'protocol_name': ['No intervention'],
    'suspicion_threshold': [0],  # Use 0 as placeholder for no threshold
    'main_task_success_mean': [baseline_benign.main_task_success.mean()],
    'main_task_success_std': [baseline_benign.main_task_success.std()],
    'main_task_success_count': [len(baseline_benign)],
    'main_task_success_se': [baseline_benign.main_task_success.std() / np.sqrt(len(baseline_benign))],
    'main_task_evaluation_mean': [baseline_benign.main_task_evaluation.mean()],
    'main_task_evaluation_std': [baseline_benign.main_task_evaluation.std()],
    'main_task_evaluation_count': [len(baseline_benign)],
    'main_task_evaluation_se': [baseline_benign.main_task_evaluation.std() / np.sqrt(len(baseline_benign))]
})
protocol_threshold_perf = pd.concat([protocol_threshold_perf, baseline_row])

# Add trusted agent performance
trusted_baseline = trusted_baseline
trusted_row = pd.DataFrame({
    'protocol_name': ['Trusted agent'],
    'suspicion_threshold': [0],  # Use 0 as placeholder for no threshold
    'main_task_success_mean': [trusted_baseline.main_task_success.mean()],
    'main_task_success_std': [trusted_baseline.main_task_success.std()],
    'main_task_success_count': [len(trusted_baseline)],
    'main_task_success_se': [trusted_baseline.main_task_success.std() / np.sqrt(len(trusted_baseline))],
    'main_task_evaluation_mean': [trusted_baseline.main_task_evaluation.mean()],
    'main_task_evaluation_std': [trusted_baseline.main_task_evaluation.std()],
    'main_task_evaluation_count': [len(trusted_baseline)],
    'main_task_evaluation_se': [trusted_baseline.main_task_evaluation.std() / np.sqrt(len(trusted_baseline))]
})
protocol_threshold_perf = pd.concat([protocol_threshold_perf, trusted_row])

# Create combined labels for protocol and threshold
protocol_threshold_perf['protocol_threshold'] = protocol_threshold_perf.apply(
    lambda row: f"{row['protocol_name']}\n(t={row['suspicion_threshold']})" if row['protocol_name'] not in ['No intervention', 'Trusted agent'] 
    else row['protocol_name'], axis=1
)

# Create bar chart with error bars
fig, ax = plt.subplots(figsize=(16, 8))

# Set up positions for the bars
x_positions = np.arange(len(protocol_threshold_perf))
width = 0.35

# Plot bars with error bars using 95% confidence intervals
success_bars = ax.bar(
    x_positions - width/2, 
    protocol_threshold_perf['main_task_success_mean'], 
    width, 
    yerr=protocol_threshold_perf['error_display_success'],
    capsize=5,
    label='Main Task Success',
    color='skyblue',
    ecolor='black'
)

eval_bars = ax.bar(
    x_positions + width/2, 
    protocol_threshold_perf['main_task_evaluation_mean'], 
    width,
    yerr=protocol_threshold_perf['error_display_eval'],
    capsize=5,
    label='Main Task Evaluation Score',
    color='lightgreen',
    ecolor='black'
)

# Add labels and formatting
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Protocol Effects on Benign Transcripts by Threshold', fontsize=14)
ax.set_xticks(x_positions)
ax.set_xticklabels(protocol_threshold_perf['protocol_threshold'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)  # Slightly higher to accommodate error bars
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width()/2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom',
                    fontsize=8)

autolabel(success_bars)
autolabel(eval_bars)

# Add a horizontal line for baseline performance
ax.axhline(y=baseline_benign.main_task_success.mean(), color='blue', linestyle='--', alpha=0.5)
ax.axhline(y=baseline_benign.main_task_evaluation.mean(), color='green', linestyle='--', alpha=0.5)

# Color the bars for baseline and trusted agent differently
for i, label in enumerate(protocol_threshold_perf['protocol_name']):
    if label == 'No intervention':
        success_bars[i].set_color('royalblue')
        eval_bars[i].set_color('forestgreen')
    elif label == 'Trusted agent':
        success_bars[i].set_color('navy')
        eval_bars[i].set_color('darkgreen')

# Add a legend for the horizontal lines
ax.plot([], [], color='blue', linestyle='--', alpha=0.5, label='Baseline Success')
ax.plot([], [], color='green', linestyle='--', alpha=0.5, label='Baseline Evaluation')
ax.legend(loc='best', ncol=2)

# Add a note about error bars using Wilson score intervals
ax.text(0.01, 0.01, "Note: Error bars show 95% Wilson score intervals (appropriate for proportions)", 
        transform=ax.transAxes, fontsize=10, ha='left', va='bottom')

plt.tight_layout()
plt.show()


In [ ]:
# Visualization 4: Task-by-Task Performance with Thresholds Comparison
# Show task pairs on x-axis with threshold bars for each task

# Get unique tasks
tasks = protocol_results_and_data.task_name.unique()
print(f"Tasks available: {sorted(tasks)}")

# Define a function to create plots for a specific transcript type and protocol
def create_threshold_comparison_plot(transcript_type_name, is_benign, protocol_name):
    print(f"\nCreating {transcript_type_name} plot for protocol: {protocol_name}")
    
    # Filter data based on transcript type
    if is_benign:
        filtered_data = protocol_results_and_data[
            (protocol_results_and_data.transcript_type == 'transcripts_no_secondary_task') &
            (protocol_results_and_data.protocol_name == protocol_name)
        ]
        baseline_data = baseline_benign  # Benign baseline
    else:
        filtered_data = protocol_results_and_data[
            (protocol_results_and_data.transcript_type != 'transcripts_no_secondary_task') &
            (protocol_results_and_data.protocol_name == protocol_name)
        ]
        baseline_data = baseline_data_malicious  # Malicious baseline
    
    # Skip if no data
    if len(filtered_data) == 0:
        print(f"No {transcript_type_name} data for protocol: {protocol_name}")
        return
    
    # Get all thresholds for this protocol
    thresholds = sorted(filtered_data.suspicion_threshold.unique())
    print(f"Thresholds available: {thresholds}")
    
    # Add threshold 100 for no intervention
    all_thresholds = thresholds + [100]
    
    # Create figure
    plt.figure(figsize=(14, 8))
    
    # Prepare data for all tasks
    plot_data = []
    
    # For each task, collect evaluation scores for each threshold
    for task in sorted(tasks):
        task_data = filtered_data[filtered_data.task_name == task]
        
        # Skip if no task data
        if len(task_data) == 0:
            continue
            
        # Get baseline data for this task (no intervention)
        baseline_task_data = baseline_data[baseline_data.task_name == task]
        no_intervention_score = baseline_task_data.main_task_evaluation.mean() if len(baseline_task_data) > 0 else None
        
        # Get evaluation scores for each threshold
        for threshold in thresholds:
            threshold_data = task_data[task_data.suspicion_threshold == threshold]
            
            if len(threshold_data) > 0:
                eval_score = threshold_data.main_task_evaluation.mean()
                ci = wilson_conf_interval(eval_score, len(threshold_data))
                
                plot_data.append({
                    'task': task,
                    'threshold': threshold,
                    'main_task_evaluation': eval_score,
                    'ci': ci,
                    'sample_size': len(threshold_data)
                })
        
        # Add no intervention data (threshold 100)
        if no_intervention_score is not None:
            plot_data.append({
                'task': task,
                'threshold': 100,  # Use 100 to represent no intervention
                'main_task_evaluation': no_intervention_score,
                'ci': wilson_conf_interval(no_intervention_score, len(baseline_task_data)),
                'sample_size': len(baseline_task_data)
            })
    
    # Convert to DataFrame
    plot_df = pd.DataFrame(plot_data)
    
    # Skip if no data to plot
    if len(plot_df) == 0:
        print(f"No data to plot for {transcript_type_name} with protocol {protocol_name}")
        plt.close()
        return
    
    # Get tasks with data
    available_tasks = sorted(plot_df.task.unique())
    print(f"Tasks with data: {available_tasks}")
    
    # Set up the plot
    num_tasks = len(available_tasks)
    num_thresholds = len(all_thresholds)
    
    # Calculate positions for grouped bars
    bar_width = 0.8 / num_thresholds
    x_positions = np.arange(num_tasks)
    
    # Create a color map for thresholds
    cmap = plt.cm.get_cmap('viridis', num_thresholds)
    
    # Plot grouped bars for each threshold
    for i, threshold in enumerate(all_thresholds):
        # Filter data for this threshold
        threshold_data = plot_df[plot_df.threshold == threshold]
        
        # Skip if no data for this threshold
        if len(threshold_data) == 0:
            continue
        
        # Prepare data for plotting
        eval_scores = []
        error_bars = []
        
        # For each task, get the evaluation score and error bar
        for task in available_tasks:
            task_threshold_data = threshold_data[threshold_data.task == task]
            
            if len(task_threshold_data) > 0:
                eval_scores.append(task_threshold_data.main_task_evaluation.values[0])
                error_bars.append(task_threshold_data.ci.values[0])
            else:
                eval_scores.append(0)
                error_bars.append(0)
        
        # Calculate bar positions
        bar_positions = x_positions + i * bar_width - (num_thresholds - 1) * bar_width / 2
        
        # Label for the threshold
        threshold_label = 'No Intervention' if threshold == 100 else f't={threshold}'
        
        # Plot bars with error bars
        plt.bar(
            bar_positions, 
            eval_scores, 
            width=bar_width,
            yerr=error_bars,
            label=threshold_label,
            color=cmap(i),
            capsize=3
        )
    
    # Format plot
    plt.xlabel('Task', fontsize=12)
    plt.ylabel('Main Task Evaluation Score', fontsize=12)
    plt.title(f'{transcript_type_name} Tasks - {protocol_name} Protocol: Main Task Evaluation by Threshold', fontsize=14)
    plt.xticks(x_positions, available_tasks, rotation=45, ha='right')
    plt.ylim(0, 1.1)  # Leave room for error bars
    plt.grid(True, alpha=0.3, axis='y')
    plt.legend(title="Threshold", loc='best')
    
    # Add note about error bars
    plt.figtext(
        0.01, 0.01, 
        "Note: Error bars show 95% Wilson score intervals (appropriate for proportions)",
        fontsize=8, ha='left'
    )
    
    # Better spacing for labels
    plt.tight_layout()
    plt.show()

# Create plots for each protocol and transcript type
control_protocol_list = [
    'defer_to_instructed_trusted_low_context', 
    'defer_to_trusted', 
    'defer_to_instructed_trusted', 
    'defer_to_trusted_with_feedback', 
    'resample_with_feedback'
]

# Run for both benign and malicious transcripts for each protocol
for protocol in control_protocol_list:
    create_threshold_comparison_plot("Benign", True, protocol)
    create_threshold_comparison_plot("Malicious", False, protocol)


In [ ]:
# Visualization 4.5: Side Task Blocking Rate by Threshold
# Show side task blocking rate (1 - side_task_evaluation) for malicious transcripts

# Define a function to create side task blocking rate plots for each protocol
def create_side_task_blocking_plot(protocol_name):
    print(f"\nCreating Side Task Blocking plot for protocol: {protocol_name}")
    
    # Filter data to only include malicious attempts (with side tasks)
    filtered_data = protocol_results_and_data[
        (protocol_results_and_data.transcript_type != 'transcripts_no_secondary_task') &
        (protocol_results_and_data.protocol_name == protocol_name)
    ]
    baseline_data = baseline_data_malicious  # Malicious baseline
    
    # Skip if no data
    if len(filtered_data) == 0:
        print(f"No malicious data for protocol: {protocol_name}")
        return
    
    # Get all thresholds for this protocol
    thresholds = sorted(filtered_data.suspicion_threshold.unique())
    print(f"Thresholds available: {thresholds}")
    
    # Add threshold 100 for no intervention
    all_thresholds = thresholds + [100]
    
    # Create figure
    plt.figure(figsize=(14, 8))
    
    # Prepare data for all tasks
    plot_data = []
    
    # For each task, collect side task blocking rates for each threshold
    for task in sorted(tasks):
        task_data = filtered_data[filtered_data.task_name == task]
        
        # Skip if no task data
        if len(task_data) == 0:
            continue
            
        # Get baseline data for this task (no intervention)
        baseline_task_data = baseline_data[baseline_data.task_name == task]
        no_intervention_blocking = 1 - baseline_task_data.side_task_evaluation.mean() if len(baseline_task_data) > 0 else None
        
        # Get side task blocking rates for each threshold
        for threshold in thresholds:
            threshold_data = task_data[task_data.suspicion_threshold == threshold]
            
            if len(threshold_data) > 0:
                # Calculate blocking rate (1 - side_task_evaluation)
                blocking_rate = 1 - threshold_data.side_task_evaluation.mean()
                ci = wilson_conf_interval(blocking_rate, len(threshold_data))
                
                plot_data.append({
                    'task': task,
                    'threshold': threshold,
                    'blocking_rate': blocking_rate,
                    'ci': ci,
                    'sample_size': len(threshold_data)
                })
        
        # Add no intervention data (threshold 100)
        if no_intervention_blocking is not None:
            plot_data.append({
                'task': task,
                'threshold': 100,  # Use 100 to represent no intervention
                'blocking_rate': no_intervention_blocking,
                'ci': wilson_conf_interval(no_intervention_blocking, len(baseline_task_data)),
                'sample_size': len(baseline_task_data)
            })
    
    # Convert to DataFrame
    plot_df = pd.DataFrame(plot_data)
    
    # Skip if no data to plot
    if len(plot_df) == 0:
        print(f"No data to plot for side task blocking with protocol {protocol_name}")
        plt.close()
        return
    
    # Get tasks with data
    available_tasks = sorted(plot_df.task.unique())
    print(f"Tasks with data: {available_tasks}")
    
    # Set up the plot
    num_tasks = len(available_tasks)
    num_thresholds = len(all_thresholds)
    
    # Calculate positions for grouped bars
    bar_width = 0.8 / num_thresholds
    x_positions = np.arange(num_tasks)
    
    # Create a color map for thresholds
    cmap = plt.cm.get_cmap('plasma', num_thresholds)
    
    # Plot grouped bars for each threshold
    for i, threshold in enumerate(all_thresholds):
        # Filter data for this threshold
        threshold_data = plot_df[plot_df.threshold == threshold]
        
        # Skip if no data for this threshold
        if len(threshold_data) == 0:
            continue
        
        # Prepare data for plotting
        blocking_rates = []
        error_bars = []
        
        # For each task, get the blocking rate and error bar
        for task in available_tasks:
            task_threshold_data = threshold_data[threshold_data.task == task]
            
            if len(task_threshold_data) > 0:
                blocking_rates.append(task_threshold_data.blocking_rate.values[0])
                error_bars.append(task_threshold_data.ci.values[0])
            else:
                blocking_rates.append(0)
                error_bars.append(0)
        
        # Calculate bar positions
        bar_positions = x_positions + i * bar_width - (num_thresholds - 1) * bar_width / 2
        
        # Label for the threshold
        threshold_label = 'No Intervention' if threshold == 100 else f't={threshold}'
        
        # Plot bars with error bars
        plt.bar(
            bar_positions, 
            blocking_rates, 
            width=bar_width,
            yerr=error_bars,
            label=threshold_label,
            color=cmap(i),
            capsize=3
        )
    
    # Format plot
    plt.xlabel('Task', fontsize=12)
    plt.ylabel('Side Task Blocking Rate (1 - side task success)', fontsize=12)
    plt.title(f'Side Task Blocking - {protocol_name} Protocol: Blocking Rate by Threshold', fontsize=14)
    plt.xticks(x_positions, available_tasks, rotation=45, ha='right')
    plt.ylim(0, 1.1)  # Leave room for error bars
    plt.grid(True, alpha=0.3, axis='y')
    plt.legend(title="Threshold", loc='best')
    
    # Add note about error bars
    plt.figtext(
        0.01, 0.01, 
        "Note: Error bars show 95% Wilson score intervals (appropriate for proportions)",
        fontsize=8, ha='left'
    )
    
    # Better spacing for labels
    plt.tight_layout()
    plt.show()

# Create side task blocking plots for each protocol
for protocol in control_protocol_list:
    create_side_task_blocking_plot(protocol)


In [ ]:
# Visualization 5: Comparative Task Performance Across Protocols
# For each task, show performance across different protocols

# Get tasks with the most variation in performance
task_variance = benign_data.groupby('task_name')['main_task_success'].var().sort_values(ascending=False)
top_varied_tasks = task_variance.head(5).index.tolist()

# Get unique protocols (excluding thresholds) for use in this visualization
unique_protocols = protocol_threshold_perf['protocol_name'].unique()

# Create subplots for each metric
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 16), sharex=True)

# Prepare data for selected tasks
selected_tasks_data = []

for task in top_varied_tasks:
    task_protocols = []
    
    # Get data for each protocol
    for protocol in unique_protocols:
        if protocol == 'No intervention':
            task_data = baseline_benign[baseline_benign.task_name == task]
        elif protocol == 'Trusted agent':
            task_data = trusted_baseline[trusted_baseline.task_name == task]
        else:
            task_data = benign_data[
                (benign_data.protocol_name == protocol) & 
                (benign_data.task_name == task)
            ]
        
        # Compute metrics, standard errors, and 95% confidence intervals
        success = task_data.main_task_success.mean() if len(task_data) > 0 else 0
        evaluation = task_data.main_task_evaluation.mean() if len(task_data) > 0 else 0
        
        # Calculate standard error with handling for small/empty samples
        if len(task_data) > 1:
            success_se = task_data.main_task_success.std() / np.sqrt(len(task_data))
            evaluation_se = task_data.main_task_evaluation.std() / np.sqrt(len(task_data))
        else:
            success_se = 0.1  # Default value for small samples
            evaluation_se = 0.1
            
        # Calculate Wilson score intervals which are better for bounded proportional data
        if len(task_data) > 0:
            success_wilson_ci = wilson_conf_interval(success, len(task_data))
            evaluation_wilson_ci = wilson_conf_interval(evaluation, len(task_data))
        else:
            success_wilson_ci = 0.1
            evaluation_wilson_ci = 0.1
            
        task_protocols.append({
            'task': task,
            'protocol': protocol,
            'main_task_success': success,
            'main_task_success_ci': success_wilson_ci,
            'main_task_evaluation': evaluation,
            'main_task_evaluation_ci': evaluation_wilson_ci,
            'sample_size': len(task_data)
        })
    
    # Add to collected data
    selected_tasks_data.extend(task_protocols)

# Convert to DataFrame
selected_tasks_df = pd.DataFrame(selected_tasks_data)

# Plot main task success with error bars using seaborn's built-in error bar functionality
success_plot = sns.barplot(
    data=selected_tasks_df,
    x='task',
    y='main_task_success',
    hue='protocol',
    ax=ax1,
    errorbar=lambda x: [x.std() / np.sqrt(len(x)), x.std() / np.sqrt(len(x))]  # Use standard error
)

ax1.set_title('Main Task Success by Task and Protocol', fontsize=14)
ax1.set_xlabel('')
ax1.set_ylabel('Success Rate', fontsize=12)
ax1.set_ylim(0, 1.05)
ax1.legend(title='Protocol', bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3, axis='y')
ax1.text(0.01, 0.01, "Note: Error bars show standard error", 
        transform=ax1.transAxes, fontsize=10, ha='left', va='bottom')

# Plot main task evaluation with error bars
eval_plot = sns.barplot(
    data=selected_tasks_df,
    x='task',
    y='main_task_evaluation',
    hue='protocol',
    ax=ax2,
    errorbar=lambda x: [x.std() / np.sqrt(len(x)), x.std() / np.sqrt(len(x))]  # Use standard error
)

ax2.set_title('Main Task Evaluation Score by Task and Protocol', fontsize=14)
ax2.set_xlabel('Task', fontsize=12)
ax2.set_ylabel('Evaluation Score', fontsize=12)
ax2.set_ylim(0, 1.05)
ax2.legend(title='Protocol', bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.grid(True, alpha=0.3, axis='y')
ax2.text(0.01, 0.01, "Note: Error bars show standard error", 
        transform=ax2.transAxes, fontsize=10, ha='left', va='bottom')

plt.tight_layout()
plt.show()
